In [1]:
from datasets import load_dataset
from itertools import islice

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.simple",
    split="train",
    streaming=True
)

articles = list(islice(dataset, 500))

print("Loaded:", len(articles))
print("Columns:", articles[0].keys())

for article in articles:
    print("\n" + "=" * 60)
    print("TITLE:", article["title"])
    print("TEXT:")
    print(article["text"][:1500])


README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

Loaded: 500
Columns: dict_keys(['id', 'url', 'title', 'text'])

TITLE: April
TEXT:
April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.

The Month 

April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.

April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.

In common years, April starts on the same day of the week as October of the previous year, and in leap years, May of the previous y

In [2]:
# Cell 2: Inspect full articles

titles = [
    "April",
    "Art",
    "Alan Turing",
    "Adobe Illustrator"
]

for title in titles:
    article = next(
        a for a in articles
        if a["title"] == title
    )

    print("\n" + "=" * 70)
    print("ARTICLE:", title)
    print("=" * 70)

    lines = article["text"].splitlines()

    for i, line in enumerate(lines, 1):
        print(f"{i:4}: {line}")

    print("\nTOTAL LINES:", len(lines))



ARTICLE: April
   1: April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.
   2: 
   3: April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.
   4: 
   5: The Month 
   6: 
   7: April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.
   8: 
   9: April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.
  10: 
  11: In common years, April starts on the same day of the week as October of the previous year, and in leap years, May of the previous ye

In [3]:
from pathlib import Path
import sys

for folder in [
    Path.cwd(),
    Path.cwd() / "backend/notebooks/modern"
]:
    if (folder / "modern_english_cleaning.py").exists():
        sys.path.insert(0, str(folder))
        break

from modern_english_cleaning import (
    clean_article,
    validate_article
)

for article in articles[:4]:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    print("=" * 60)
    print("ARTICLE:", article["title"])
    print("REMOVED:", report["removed_pct"], "%")
    print("FLAGS:", flags)
    print("\nCLEANED TEXT:")
    print(cleaned[:1000])

ARTICLE: April
REMOVED: 2.01 %
FLAGS: []

CLEANED TEXT:
April (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of the four months to have 30 days.

April always begins on the same day of the week as July, and additionally, January in leap years. April always ends on the same day of the week as December.

April comes between March and May, making it the fourth month of the year. It also comes first in the year out of the four months that have 30 days, as June, September and November are later in the year.

April begins on the same day of the week as July every year and on the same day of the week as January in leap years. April ends on the same day of the week as December every year, as each other's last days are exactly 35 weeks (245 days) apart.

In common years, April starts on the same day of the week as October of the previous year, and in leap years, May of the previous year. In common years, April finishes on

In [4]:
# Inspect the endings of cleaned articles

for article in articles[:10]:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    print("=" * 60)
    print("ARTICLE:", article["title"])

    print("\nORIGINAL END:")
    print(article["text"][-500:])

    print("\nCLEANED END:")
    print(cleaned[-500:])

    print("\nREMOVED:", report["removed_pct"], "%")
    print("FLAGS:", flags)


ARTICLE: April

ORIGINAL END:
e arranged in alphabetical order in the English language, April would come first.
 Five current European monarchs were born in April. They are King Philippe of Belgium (April 15), Queen Margrethe II of Denmark (April 16), Henri, Grand Duke of Luxembourg (April 16), King Willem-Alexander of the Netherlands (April 27), and King Carl XVI Gustaf of Sweden (April 30). Elizabeth II of the United Kingdom and Commonwealth realms - who died on September 8, 2022 - was also born in April (on April 21).

04

CLEANED END:
r were arranged in alphabetical order in the English language, April would come first.
Five current European monarchs were born in April. They are King Philippe of Belgium (April 15), Queen Margrethe II of Denmark (April 16), Henri, Grand Duke of Luxembourg (April 16), King Willem-Alexander of the Netherlands (April 27), and King Carl XVI Gustaf of Sweden (April 30). Elizabeth II of the United Kingdom and Commonwealth realms - who died on September 8,

In [5]:

from collections import Counter

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "removed_pct": report["removed_pct"],
        "flags": flags
    })

print("Total articles:", len(results))
print("Flagged articles:", sum(bool(r["flags"]) for r in results))
print("Empty articles:", sum(not r["cleaned"] for r in results))

flag_counts = Counter(
    flag for r in results for flag in r["flags"]
)

print("\nFLAG COUNTS:")
for flag, count in flag_counts.most_common():
    print(f"{flag}: {count}")

print("\nTOP 10 MOST MODIFIED:")

most_modified = sorted(
    results,
    key=lambda r: r["removed_pct"],
    reverse=True
)

for article in most_modified[:10]:
    print(
        f"{article['title'][:35]:35} "
        f"{article['removed_pct']:.2f}% "
        f"{article['flags']}"
    )


Total articles: 500
Flagged articles: 105
Empty articles: 0

FLAG COUNTS:
possible_trailing_category: 69
empty_parentheses: 25
very_short: 10
large_text_removal: 6
leftover_wiki_markup: 1

TOP 10 MOST MODIFIED:
Historian                           93.88% ['very_short', 'large_text_removal']
Healing                             60.30% ['large_text_removal']
List of mathematics topics          46.57% ['large_text_removal']
Southern Ocean                      46.01% ['large_text_removal']
Saint Lawrence River                45.49% ['large_text_removal']
Indian Ocean                        43.79% ['large_text_removal']
South America                       38.02% []
Goatee                              36.73% ['very_short']
Ethics                              34.18% []
Network                             31.61% ['very_short']


In [6]:

# Cell 4: Investigate heavily cleaned articles

titles_to_check = [
    "Historian",
    "Healing",
    "May",
    "List of mathematics topics",
    "Southern Ocean",
    "Saint Lawrence River",
    "Indian Ocean"
]

for title in titles_to_check:
    article = next(
        (a for a in articles if a["title"] == title),
        None
    )

    if article is None:
        continue

    original = article["text"]

    cleaned, report = clean_article(
        original,
        title=title
    )

    print("\n" + "=" * 70)
    print("ARTICLE:", title)
    print("REMOVED:", report["removed_pct"], "%")
    print("STOPPED AT:", report["stopped_at_back_matter"])

    print("\n=== END OF CLEANED TEXT ===")
    print(cleaned[-700:])

    # Find where the cleaning stopped
    lines = original.splitlines()
    section = report["stopped_at_back_matter"]

    if section:
        index = next(
            (
                i for i, line in enumerate(lines)
                if line.strip().rstrip(":").casefold()
                == section
            ),
            None
        )

        if index is not None:
            print("\n=== ORIGINAL AROUND STOP POINT ===")

            start = max(0, index - 5)
            end = min(len(lines), index + 15)

            for j in range(start, end):
                print(f"{j+1}: {lines[j][:200]}")

            print(
                "\nLINES AFTER STOP:",
                len(lines) - index - 1
            )

    print("\n=== ORIGINAL END ===")
    print(original[-700:])



ARTICLE: Historian
REMOVED: 93.88 %
STOPPED AT: related pages

=== END OF CLEANED TEXT ===
A historian is someone who studies history. Historians use written sources to understand past events and societies.

=== ORIGINAL AROUND STOP POINT ===
1: A historian is someone who studies history.  Historians use written sources to understand past events and societies.
2: 
3: Related pages
4: List of historians
5: 
6: References
7: 
8: Sources 
9:  Richard B. Todd, ed. (2004). Dictionary of British Classicists, 1500–1960, Bristol: Thoemmes Continuum, 2004 .
10:  Kelly Boyd, ed. (1999). Encyclopedia of Historians and Historical Writing. London [etc.] : Fitzroy Dearborn 
11:  Lateiner, D. (1989). The historical method of Herodotus. Phoenix, 23. Toronto: University of Toronto Press.
12:  John Cannon et al., eds. (1988). The Blackwell Dictionary of Historians. Oxford: Blackwell Publishers, 1988 .
13:  Hartog, F. (1988). The mirror of Herodotus: the representation of the other in the writing of his

In [7]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article

for title in ["May", "Historian", "Healing"]:
    article = next(
        a for a in articles if a["title"] == title
    )

    cleaned, report = clean_article(
        article["text"],
        title=title
    )

    print("\nARTICLE:", title)
    print("REMOVED:", report["removed_pct"], "%")

    if title == "May":
        print("Historical events preserved:",
              "Great Exhibition" in cleaned)

    if title == "Historian":
        print("References removed:",
              "Dictionary of British Classicists" not in cleaned)


ARTICLE: May
REMOVED: 1.95 %
Historical events preserved: True

ARTICLE: Historian
REMOVED: 93.88 %
References removed: True

ARTICLE: Healing
REMOVED: 60.3 %


In [8]:
from collections import Counter

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "removed_pct": report["removed_pct"],
        "flags": flags
    })

print("Total articles:", len(results))
print("Flagged articles:", sum(bool(r["flags"]) for r in results))
print("Empty articles:", sum(not r["cleaned"] for r in results))

print("\nFLAG COUNTS:")
for flag, count in Counter(
    flag for r in results for flag in r["flags"]
).most_common():
    print(flag, count)

print("\nTOP 10 MOST MODIFIED:")
for r in sorted(
    results,
    key=lambda x: x["removed_pct"],
    reverse=True
)[:10]:
    print(
        f"{r['title'][:35]:35} "
        f"{r['removed_pct']:.2f}% "
        f"{r['flags']}"
    )

Total articles: 500
Flagged articles: 105
Empty articles: 0

FLAG COUNTS:
possible_trailing_category 69
empty_parentheses 25
very_short 10
large_text_removal 6
leftover_wiki_markup 1

TOP 10 MOST MODIFIED:
Historian                           93.88% ['very_short', 'large_text_removal']
Healing                             60.30% ['large_text_removal']
List of mathematics topics          46.57% ['large_text_removal']
Southern Ocean                      46.01% ['large_text_removal']
Saint Lawrence River                45.49% ['large_text_removal']
Indian Ocean                        43.79% ['large_text_removal']
South America                       38.02% []
Goatee                              36.73% ['very_short']
Ethics                              34.18% []
Network                             31.61% ['very_short']


In [9]:

# Inspect remaining cleaning problems

import re

# Patterns for leftover Wikipedia markup
MARKUP_PATTERN = re.compile(
    r"\[\[|\]\]|\{\{|\}\}|<ref\b",
    re.I
)

for result in results:

    relevant_flags = {
        "possible_trailing_metadata",
        "leftover_wiki_markup"
    }

    if not relevant_flags.intersection(result["flags"]):
        continue

    print("\n" + "=" * 65)
    print("ARTICLE:", result["title"])
    print("FLAGS:", result["flags"])
    print("REMOVED:", result["removed_pct"], "%")

    print("\n=== ORIGINAL LAST 8 LINES ===")
    print("\n".join(result["original"].splitlines()[-8:]))

    print("\n=== CLEANED LAST 8 LINES ===")
    print("\n".join(result["cleaned"].splitlines()[-8:]))

    # Show remaining markup and its context
    matches = list(MARKUP_PATTERN.finditer(result["cleaned"]))

    if matches:
        print("\n=== LEFTOVER MARKUP ===")

        for match in matches[:5]:
            start = max(0, match.start() - 60)
            end = min(len(result["cleaned"]), match.end() + 60)

            print(repr(result["cleaned"][start:end]))



ARTICLE: Embassy
FLAGS: ['leftover_wiki_markup', 'possible_trailing_category']
REMOVED: 1.02 %

=== ORIGINAL LAST 8 LINES ===
A foreign embassy is the official office of one country in another. It is usually in the capital city of the other country. It is where the [[ambassador]relationship between the two countries. 

The head of the embassy is usually an ambassador, but can also be a minister, high commissioner, or other level of diplomatic personnel appointed by the sending country to represent it.

 

Diplomacy

=== CLEANED LAST 8 LINES ===
A foreign embassy is the official office of one country in another. It is usually in the capital city of the other country. It is where the [[ambassador]relationship between the two countries.

The head of the embassy is usually an ambassador, but can also be a minister, high commissioner, or other level of diplomatic personnel appointed by the sending country to represent it.

Diplomacy

=== LEFTOVER MARKUP ===
'y in the capital city of the ot

In [10]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article

for title in ["April", "May", "December"]:
    article = next(a for a in articles if a["title"] == title)
    cleaned, report = clean_article(article["text"], title=title)

    print(f"\nARTICLE: {title}")
    print("Last line:", repr(cleaned.splitlines()[-1]))
    print("Removed:", report["removed_pct"], "%")


ARTICLE: April
Last line: 'Five current European monarchs were born in April. They are King Philippe of Belgium (April 15), Queen Margrethe II of Denmark (April 16), Henri, Grand Duke of Luxembourg (April 16), King Willem-Alexander of the Netherlands (April 27), and King Carl XVI Gustaf of Sweden (April 30). Elizabeth II of the United Kingdom and Commonwealth realms - who died on September 8, 2022 - was also born in April (on April 21).'
Removed: 2.01 %

ARTICLE: May
Last line: 'No President of the United States, to-date, has died in May, though James Buchanan narrowly avoided doing so, dying on the morning of June 1, 1868.'
Removed: 1.95 %

ARTICLE: December
Last line: 'At the North Pole, the Sun does not rise in December; at the South Pole, it does not set.'
Removed: 1.27 %


In [11]:
import re

titles = [
    "Embassy",
    "Earth science",
    "Experiment",
    "South America"
]

for title in titles:
    article = next(a for a in articles if a["title"] == title)
    original = article["text"]

    print("\n" + "=" * 60)
    print("ARTICLE:", title)

    matches = list(re.finditer(
        r"\[\[|<ref\b", original, re.I
    ))

    for match in matches[:5]:
        start = max(0, match.start() - 100)
        end = min(len(original), match.start() + 500)

        print("\nMARKUP CONTEXT:")
        print(repr(original[start:end]))


ARTICLE: Embassy

MARKUP CONTEXT:
' of one country in another. It is usually in the capital city of the other country. It is where the [[ambassador]relationship between the two countries. \n\nThe head of the embassy is usually an ambassador, but can also be a minister, high commissioner, or other level of diplomatic personnel appointed by the sending country to represent it.\n\n \n\nDiplomacy'

ARTICLE: Earth science

MARKUP CONTEXT:
', Mineralogy, petrophysics, stratigraphy, structural geology, engineering geology and sedimentology.<ref\nname="smith 5">Smith, Gary A. & Pun, Aurora 2006. How does the Earth work?. Pearson Prentice Hall, NJ. .</ref>\n Geophysics and Geodesy investigate the shape of the Earth, its reaction to forces and its magnetic and gravity fields. Geophysicists explore the Earth\'s core and mantle as well as the tectonic and seismic activity of the lithosphere.\n Soil science covers the outermost layer of the Earth\'s crust that is subject to soil formation processe

In [12]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article
validate_article = modern_english_cleaning.validate_article

for title in [
    "Earth science",
    "Experiment",
    "South America",
    "Embassy"
]:
    article = next(
        a for a in articles if a["title"] == title
    )

    cleaned, report = clean_article(
        article["text"],
        title=title
    )

    flags = validate_article(cleaned, report)

    print("\nARTICLE:", title)
    print("Remaining <ref>:", "<ref" in cleaned.lower())
    print("FLAGS:", flags)


ARTICLE: Earth science
Remaining <ref>: False
FLAGS: []

ARTICLE: Experiment
Remaining <ref>: False
FLAGS: []

ARTICLE: South America
Remaining <ref>: False
FLAGS: []

ARTICLE: Embassy
Remaining <ref>: False
FLAGS: ['leftover_wiki_markup', 'possible_trailing_category']


In [13]:

import importlib
import modern_english_cleaning
from collections import Counter

# Reload the latest cleaning functions
importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article
validate_article = modern_english_cleaning.validate_article

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "removed_pct": report["removed_pct"],
        "flags": flags
    })

print("TOTAL ARTICLES:", len(results))
print("FLAGGED:", sum(bool(r["flags"]) for r in results))
print("EMPTY:", sum(not r["cleaned"] for r in results))

print("\nFLAG COUNTS:")
counts = Counter(
    flag for r in results for flag in r["flags"]
)

for flag, count in counts.most_common():
    print(f"{flag}: {count}")

print("\nREMAINING MARKUP ISSUES:")
for r in results:
    if "leftover_wiki_markup" in r["flags"]:
        print(r["title"])

# Verify meaningful content was preserved
south_america = next(
    r for r in results
    if r["title"] == "South America"
)

print("\nSOUTH AMERICA CHECK:")
print(
    "Panama Canal sentence preserved:",
    "These two continents are separated by the Panama Canal."
    in south_america["cleaned"]
)



TOTAL ARTICLES: 500
FLAGGED: 105
EMPTY: 0

FLAG COUNTS:
possible_trailing_category: 69
empty_parentheses: 25
very_short: 10
large_text_removal: 6
leftover_wiki_markup: 1

REMAINING MARKUP ISSUES:
Embassy

SOUTH AMERICA CHECK:
Panama Canal sentence preserved: True


In [14]:

# Final cleaning review

flagged = [r for r in results if r["flags"]]

print("FLAGGED ARTICLES:", len(flagged))

for r in flagged:
    print("\n" + "=" * 60)
    print("TITLE:", r["title"])
    print("FLAGS:", r["flags"])
    print("REMOVED:", r["removed_pct"], "%")

    print("\nCLEANED TEXT (last 400 chars):")
    print(r["cleaned"][-400:])


# Check important known edge cases
print("\n" + "=" * 60)
print("KNOWN EDGE CASES")

for title in ["Andouille", "Air", "May", "Embassy"]:
    r = next(
        (item for item in results if item["title"] == title),
        None
    )

    if r is None:
        continue

    print("\nARTICLE:", title)
    print("FLAGS:", r["flags"])
    print("LAST 3 LINES:")
    print("\n".join(r["cleaned"].splitlines()[-3:]))


# Regression checks
may = next(r for r in results if r["title"] == "May")
south = next(r for r in results if r["title"] == "South America")

print("\nREGRESSION CHECKS")
print("May historical events preserved:",
      "Great Exhibition" in may["cleaned"])

print("South America Panama sentence preserved:",
      "These two continents are separated by the Panama Canal."
      in south["cleaned"])


FLAGGED ARTICLES: 105

TITLE: Andouille
FLAGS: ['possible_trailing_category']
REMOVED: 0.0 %

CLEANED TEXT (last 400 chars):
rk meat, fat, intestines (tubes going to the stomach), and tripe (the wall of the stomach).

Other sorts are "French andouille" and "German andouille"; they are less spicy than Cajun. Cajun has extra salt, black pepper, and garlic. Andouille makers smoke the sausages over pecan wood and sugar cane for a maximum of seven or eight hours, at about 175 degrees Fahrenheit (80 degrees Celsius).

Sausage

TITLE: Addition
FLAGS: ['possible_trailing_category']
REMOVED: 4.98 %

CLEANED TEXT (last 400 chars):
s does not change the result.

For any three numbers , , and , it is true that . For example, and , which means that .

When addition is used together with other operations, the order of operations becomes important. In the standard order of operations, addition is to be computed later than exponentiation, roots, multiplication and division, but has equal importance as

In [15]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

for title in ["Andouille", "Embassy", "Cost of living", "Italians"]:
    article = next(a for a in articles if a["title"] == title)

    cleaned, report = modern_english_cleaning.clean_article(
        article["text"],
        title=title
    )

    flags = modern_english_cleaning.validate_article(cleaned, report)

    print(title, ":", flags)

Andouille : ['possible_trailing_category']
Embassy : ['leftover_wiki_markup', 'possible_trailing_category']
Cost of living : ['very_short', 'possible_trailing_category']
Italians : ['very_short', 'empty_parentheses']


In [16]:
article = next(
    a for a in articles
    if a["title"] == "Cost of living"
)

cleaned, report = clean_article(
    article["text"],
    title=article["title"]
)

print("ORIGINAL:")
print(repr(article["text"]))

print("\nCLEANED LINES:")
for i, line in enumerate(cleaned.splitlines(), 1):
    print(i, repr(line))

ORIGINAL:
'Cost of living is the amount of money it costs just to live in a certain place.  It includes food, housing, etc.\nEconomic indicators'

CLEANED LINES:
1 'Cost of living is the amount of money it costs just to live in a certain place. It includes food, housing, etc.'
2 'Economic indicators'


In [17]:
import importlib
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

for title in ["Cost of living", "Andouille", "Embassy", "Italians"]:
    article = next(a for a in articles if a["title"] == title)

    cleaned, report = modern_english_cleaning.clean_article(
        article["text"],
        title=title
    )

    flags = modern_english_cleaning.validate_article(
        cleaned, report
    )

    print(title, ":", flags)

Cost of living : ['very_short', 'possible_trailing_category']
Andouille : ['possible_trailing_category']
Embassy : ['leftover_wiki_markup', 'possible_trailing_category']
Italians : ['very_short', 'empty_parentheses']


In [18]:

import importlib
from collections import Counter
import modern_english_cleaning

importlib.reload(modern_english_cleaning)

clean_article = modern_english_cleaning.clean_article
validate_article = modern_english_cleaning.validate_article

results = []

for article in articles:
    cleaned, report = clean_article(
        article["text"],
        title=article["title"]
    )

    flags = validate_article(cleaned, report)

    results.append({
        "title": article["title"],
        "original": article["text"],
        "cleaned": cleaned,
        "report": report,
        "flags": flags
    })

print("TOTAL:", len(results))
print("FLAGGED:", sum(bool(r["flags"]) for r in results))
print("EMPTY:", sum(not r["cleaned"] for r in results))

print("\nFLAG COUNTS:")
for flag, count in Counter(
    flag for r in results for flag in r["flags"]
).most_common():
    print(f"{flag}: {count}")

print("\nPOSSIBLE TRAILING CATEGORIES:")
for r in results:
    if "possible_trailing_category" in r["flags"]:
        print(
            f"{r['title']:35} -> "
            f"{r['cleaned'].splitlines()[-1]}"
        )

print("\nREGRESSION CHECKS:")
checks = {
    "May events preserved": "Great Exhibition" in next(
        r["cleaned"] for r in results if r["title"] == "May"
    ),
    "South America sentence preserved": "These two continents are separated by the Panama Canal." in next(
        r["cleaned"] for r in results if r["title"] == "South America"
    )
}

for name, passed in checks.items():
    print(f"{name}: {'PASS' if passed else 'FAIL'}")


TOTAL: 500
FLAGGED: 105
EMPTY: 0

FLAG COUNTS:
possible_trailing_category: 69
empty_parentheses: 25
very_short: 10
large_text_removal: 6
leftover_wiki_markup: 1

POSSIBLE TRAILING CATEGORIES:
Andouille                           -> Sausage
Addition                            -> Addition table
Abrahamic religions                 -> Mythology
Austria                             -> Gallery
Armenia                             -> Gallery
Browser                             -> Animals by eating behaviors
Beard                               -> Facial hair
Computer                            -> Largest computer companies
City                                -> Gallery of cities
Capitalization                      -> Spelling
Cube                                -> Platonic solids
Cost of living                      -> Economic indicators
Definition                          -> Words
Diesel-electric                     -> Motors
Embassy                             -> Diplomacy
Encyclopedia         

In [19]:

# Investigate empty parentheses in cleaned articles

import re

empty_parens = [
    r for r in results
    if "empty_parentheses" in r["flags"]
]

print("ARTICLES WITH EMPTY PARENTHESES:", len(empty_parens))

original_issue = 0
introduced_by_cleaning = 0

for article in empty_parens:
    original = article["original"]
    cleaned = article["cleaned"]

    original_has = bool(re.search(r"\(\s*\)", original))

    if original_has:
        original_issue += 1
        source = "ALREADY IN ORIGINAL"
    else:
        introduced_by_cleaning += 1
        source = "POSSIBLY INTRODUCED BY CLEANING"

    print("\n" + "=" * 60)
    print("ARTICLE:", article["title"])
    print("SOURCE:", source)

    match = re.search(r"\(\s*\)", cleaned)

    if match:
        start = max(0, match.start() - 100)
        end = min(len(cleaned), match.end() + 100)

        print("CONTEXT:")
        print(repr(cleaned[start:end]))

print("\n" + "=" * 60)
print("Already in original:", original_issue)
print("Possibly introduced by cleaning:", introduced_by_cleaning)


ARTICLES WITH EMPTY PARENTHESES: 25

ARTICLE: Algebra
SOURCE: ALREADY IN ORIGINAL
CONTEXT:
'he letter: . When the number is 1, then it is not written because 1 times any number is that number () and so it is not needed. And when it is 0, we can completely remove the terms, because 0 times any '

ARTICLE: Austria
SOURCE: ALREADY IN ORIGINAL
CONTEXT:
'Austria (, ; ), officially the Republic of Austria ( ), is a country in Central Europe. Around Austria there are the countries of Germany, Czech Republic,'

ARTICLE: Classical element
SOURCE: ALREADY IN ORIGINAL
CONTEXT:
' bile (earth), and blood (air).\n\nIn Chinese Taoism the elements are metal, wood, water, fire, earth ().'

ARTICLE: Christian
SOURCE: ALREADY IN ORIGINAL
CONTEXT:
'A Christian () is a person who believes in Christianity, a monotheistic religion. Christianity is mostly about the'

ARTICLE: Circle
SOURCE: ALREADY IN ORIGINAL
CONTEXT:
'uals times times ).\n\nCalculating π\ncan be measured by drawing a circle, then measuring 

In [20]:
from pathlib import Path
import json

OUTPUT_DIR = Path("modern_cleaning_output")
OUTPUT_DIR.mkdir(exist_ok=True)

# High-priority review flags
REVIEW_FLAGS = {
    "empty_parentheses",
    "leftover_wiki_markup"
}

review_count = 0

with (
    open(OUTPUT_DIR / "cleaned_articles.jsonl",
         "w", encoding="utf-8") as all_file,
    open(OUTPUT_DIR / "needs_review.jsonl",
         "w", encoding="utf-8") as review_file
):
    for article in results:
        record = {
            "title": article["title"],
            "text": article["cleaned"],
            "flags": article["flags"]
        }

        line = json.dumps(record, ensure_ascii=False) + "\n"
        all_file.write(line)

        if REVIEW_FLAGS.intersection(article["flags"]):
            review_file.write(line)
            review_count += 1

print("Total exported:", len(results))
print("Priority review articles:", review_count)
print("Saved to:", OUTPUT_DIR.resolve())

Total exported: 500
Priority review articles: 26
Saved to: /workspace/ACC-recursive-model-collapse/backend/notebooks/modern/modern_cleaning_output
